# 📊 Bloque 1 — Sesión 2: Dispersión, interpretación en ML y visualización

**Módulo:** Depuración, Limpieza y Clasificación de Datos (5109)  
**Unidad Didáctica:** UD2 — Verificación estadística de la calidad de los datos  
**Duración estimada:** ~4 horas

---

### Contenidos de esta sesión
- **1.3** Medidas de dispersión: varianza, desviación típica, rango, IQR y percentiles
- **1.4** Interpretación práctica de los estadísticos en el contexto del ML
- **1.5** Visualización: histogramas y boxplots como herramientas de diagnóstico

### Criterios de evaluación asociados
> **(CE a)** Se han descrito las técnicas estadísticas de análisis de la calidad técnica de los datos.  
> **(CE b)** Se han aplicado técnicas de evaluación de la calidad de los datos basadas en cálculos estadísticos para detectar la cobertura y el sesgo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

# Datasets
titanic = sns.load_dataset('titanic')
diamonds = sns.load_dataset('diamonds')
penguins = sns.load_dataset('penguins').dropna()

print('✅ Librerías y datasets cargados')

---
## 1.3 Medidas de dispersión

Las medidas de centralidad (media, mediana) nos dicen dónde está el centro de la distribución, pero no nos dicen **cómo de dispersos** están los datos alrededor de ese centro. Dos variables pueden tener la misma media y comportarse de manera completamente distinta.

### ¿Por qué importa la dispersión en ML?

- Una variable con **dispersión muy baja** apenas aporta información al modelo (poca varianza → baja utilidad predictiva).
- Una variable con **dispersión extrema** puede dominar distancias en algoritmos como KNN o SVM si no se normaliza.
- Detectar la dispersión real de un atributo ayuda a decidir si escalarlo, transformarlo o descartarlo.

In [ ]:
# Motivación: misma media, dispersión muy diferente
np.random.seed(42)
grupo_a = np.random.normal(loc=50, scale=2, size=200)   # poca dispersión
grupo_b = np.random.normal(loc=50, scale=15, size=200)  # mucha dispersión

print(f"Grupo A — media: {grupo_a.mean():.1f} | Grupo B — media: {grupo_b.mean():.1f}")
print("→ Misma media, comportamiento completamente distinto")

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
axes[0].hist(grupo_a, bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Grupo A — poca dispersión (σ=2)')
axes[0].axvline(grupo_a.mean(), color='crimson', linestyle='--', label='Media')
axes[0].legend()

axes[1].hist(grupo_b, bins=20, color='coral', edgecolor='white')
axes[1].set_title('Grupo B — mucha dispersión (σ=15)')
axes[1].axvline(grupo_b.mean(), color='crimson', linestyle='--', label='Media')
axes[1].legend()

plt.suptitle('Misma media, dispersión radicalmente distinta', fontsize=13)
plt.tight_layout()
plt.show()

### Varianza (s²) y Desviación Típica (s)

La **varianza** mide el promedio de los cuadrados de las diferencias entre cada valor y la media:

$$s^2 = \frac{1}{n-1} \sum_{i=1}^{n}(x_i - \bar{x})^2$$

La **desviación típica** es la raíz cuadrada de la varianza y tiene las mismas unidades que los datos:

$$s = \sqrt{s^2}$$

> 📌 Usamos `n-1` en el denominador (corrección de Bessel) porque trabajamos con una muestra, no con la población completa. Esto hace la estimación insesgada.

**Interpretación:** la desviación típica indica, en promedio, cuánto se alejan los valores de la media.

In [ ]:
# Varianza y desviación típica con pandas y numpy
edad = titanic['age'].dropna()
tarifa = titanic['fare'].dropna()

print("─── Edad ───")
print(f"  Varianza:          {edad.var():.2f}")
print(f"  Desviación típica: {edad.std():.2f} años")
print(f"  Media:             {edad.mean():.2f} años")
print(f"  → La edad típica es {edad.mean():.1f} ± {edad.std():.1f} años")

print()
print("─── Tarifa (fare) ───")
print(f"  Varianza:          {tarifa.var():.2f}")
print(f"  Desviación típica: {tarifa.std():.2f} €")
print(f"  Media:             {tarifa.mean():.2f} €")
print(f"  → Desviación >>  media: hay mucha heterogeneidad en los precios")

### Rango

El **rango** es la diferencia entre el valor máximo y el mínimo:

$$\text{Rango} = x_{\max} - x_{\min}$$

Es el estadístico de dispersión más simple, pero muy sensible a outliers. Un solo valor extremo puede dar una imagen distorsionada.

In [ ]:
for col in ['age', 'fare']:
    serie = titanic[col].dropna()
    print(f"{col}: mín={serie.min():.1f}  máx={serie.max():.1f}  rango={serie.max()-serie.min():.1f}")

### Percentiles e IQR (Rango Intercuartílico)

Los **percentiles** dividen los datos ordenados en 100 partes iguales. Los más usados son:

- **P25 (Q1):** el 25% de los datos está por debajo.
- **P50 (Q2 = mediana):** el 50% está por debajo.
- **P75 (Q3):** el 75% de los datos está por debajo.

El **IQR** (Rango Intercuartílico) es la diferencia entre Q3 y Q1:

$$\text{IQR} = Q3 - Q1$$

Representa el rango del **50% central** de los datos. Es robusto ante outliers y se usa habitualmente para detectarlos.

In [ ]:
def resumen_dispersion(serie, nombre=''):
    """Calcula un resumen completo de dispersión para una serie numérica."""
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    print(f"─── {nombre or serie.name} ───")
    print(f"  Mín:     {serie.min():.2f}")
    print(f"  Q1:      {q1:.2f}")
    print(f"  Mediana: {serie.median():.2f}")
    print(f"  Q3:      {q3:.2f}")
    print(f"  Máx:     {serie.max():.2f}")
    print(f"  Rango:   {serie.max()-serie.min():.2f}")
    print(f"  IQR:     {iqr:.2f}")
    print(f"  Std:     {serie.std():.2f}")
    print()

resumen_dispersion(titanic['age'].dropna(), 'Edad (Titanic)')
resumen_dispersion(titanic['fare'].dropna(), 'Tarifa (Titanic)')

### Detección de outliers con IQR

La regla clásica del IQR define como valores atípicos aquellos que caen fuera del intervalo:

$$[Q1 - 1.5 \cdot IQR \ , \ Q3 + 1.5 \cdot IQR]$$

Esta regla es el criterio que los **boxplots** aplican por defecto para dibujar los bigotes.

In [ ]:
def detectar_outliers_iqr(serie):
    """Devuelve la máscara booleana de outliers según el criterio IQR."""
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    lim_inf = q1 - 1.5 * iqr
    lim_sup = q3 + 1.5 * iqr
    outliers = (serie < lim_inf) | (serie > lim_sup)
    return outliers, lim_inf, lim_sup

# Aplicamos a la tarifa del Titanic
fare = titanic['fare'].dropna()
mask_out, lim_inf, lim_sup = detectar_outliers_iqr(fare)

print(f"Límite inferior: {lim_inf:.2f} €")
print(f"Límite superior: {lim_sup:.2f} €")
print(f"Outliers detectados: {mask_out.sum()} ({mask_out.mean()*100:.1f}% de la muestra)")
print()
print("Tarifas atípicas más altas:")
print(fare[mask_out].sort_values(ascending=False).head(10).values)

---
## 1.4 Interpretación de los estadísticos en el contexto del ML

Calcular estadísticos no es un fin en sí mismo. En el contexto del ML, cada estadístico nos hace una pregunta concreta sobre la utilidad y calidad del atributo:

| Estadístico | Pregunta que responde | Implicación para el modelo |
|-------------|----------------------|---------------------------|
| **Media/Mediana** | ¿Cuál es el valor central? ¿Hay sesgo de clase? | Baseline para imputación de nulos |
| **Desviación típica** | ¿Cuánta variabilidad hay? | Si ≈ 0: variable constante → descartar |
| **IQR** | ¿Cuál es el rango real sin outliers? | Referencia para escalado robusto (`RobustScaler`) |
| **Percentiles extremos** | ¿Hay valores extremos? ¿Son errores o legítimos? | Decisión de capping o eliminación |
| **Coeficiente de variación** | ¿La dispersión es grande respecto a la media? | Comparar variables con distintas escalas |

In [ ]:
# Coeficiente de variación (CV): std / media × 100
# Permite comparar la dispersión de variables con escalas distintas

cols = ['age', 'fare', 'sibsp', 'parch']
df_cv = pd.DataFrame({
    'Media': titanic[cols].mean().round(2),
    'Std': titanic[cols].std().round(2),
    'CV (%)': (titanic[cols].std() / titanic[cols].mean() * 100).round(1)
})
print(df_cv)
print()
print("→ 'fare' tiene CV > 100%: dispersión altísima relativa a la media.")
print("  Un escalado sin tener en cuenta outliers podría distorsionar el modelo.")

In [ ]:
# Variables con varianza casi nula → poco útiles para el modelo
def detectar_baja_varianza(df, umbral_cv=5.0):
    """Señala variables numéricas con coeficiente de variación menor que el umbral."""
    cols_num = df.select_dtypes(include='number').columns
    resultados = []
    for col in cols_num:
        serie = df[col].dropna()
        if serie.mean() != 0:
            cv = serie.std() / abs(serie.mean()) * 100
        else:
            cv = np.nan
        resultados.append({'Variable': col, 'CV (%)': round(cv, 1),
                           'Alerta': '⚠️ baja varianza' if cv is not np.nan and cv < umbral_cv else '✅'})
    return pd.DataFrame(resultados).set_index('Variable')

detectar_baja_varianza(titanic)

---
## 1.5 Visualización: histogramas y boxplots

La exploración visual de la distribución de cada variable es tan importante como los estadísticos numéricos. Dos representaciones complementarias son fundamentales:

### Histograma

Divide el rango de valores en intervalos (*bins*) y muestra la frecuencia de cada uno. Permite apreciar:
- La forma general de la distribución (simétrica, asimétrica, bimodal…)
- Dónde se concentran los datos
- La existencia de colas largas

### Boxplot (diagrama de caja y bigotes)

Resume la distribución en 5 estadísticos: mínimo, Q1, mediana, Q3 y máximo (sin outliers). Los puntos fuera de los bigotes son outliers según el criterio IQR. Permite comparar grupos de forma compacta.

In [ ]:
# Ejemplo integrado: histograma + boxplot para la misma variable
fig, axes = plt.subplots(2, 1, figsize=(10, 7), gridspec_kw={'height_ratios': [3, 1]})

fare = titanic['fare'].dropna()

# Histograma con curva KDE
axes[0].hist(fare, bins=40, color='steelblue', edgecolor='white', alpha=0.8, density=True)
fare.plot.kde(ax=axes[0], color='navy', linewidth=2)
axes[0].axvline(fare.mean(), color='crimson', linestyle='--', linewidth=1.5, label=f'Media: {fare.mean():.1f}')
axes[0].axvline(fare.median(), color='darkorange', linestyle='-', linewidth=1.5, label=f'Mediana: {fare.median():.1f}')
axes[0].set_xlabel('')
axes[0].set_ylabel('Densidad')
axes[0].set_title('Distribución de la tarifa (fare) — Titanic', fontsize=13)
axes[0].legend()

# Boxplot horizontal alineado
axes[1].boxplot(fare, vert=False, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6),
                medianprops=dict(color='darkorange', linewidth=2),
                flierprops=dict(marker='o', markersize=3, alpha=0.4, color='crimson'))
axes[1].set_xlabel('Tarifa (£)')
axes[1].set_yticks([])

plt.tight_layout()
plt.show()

print("→ El histograma muestra un fuerte sesgo a la derecha (cola larga hacia tarifas altas).")
print("  El boxplot revela numerosos outliers. Media >> Mediana confirma la asimetría.")

In [ ]:
# Comparación por grupos: boxplot múltiple
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Tarifa por clase social
sns.boxplot(data=titanic, x='pclass', y='fare', ax=axes[0],
            palette='Blues', order=[1, 2, 3])
axes[0].set_title('Tarifa según clase social')
axes[0].set_xlabel('Clase (1=Primera, 3=Tercera)')
axes[0].set_ylabel('Tarifa (£)')

# Edad por sexo y supervivencia
sns.boxplot(data=titanic, x='survived', y='age', hue='sex', ax=axes[1],
            palette={'male': 'steelblue', 'female': 'coral'})
axes[1].set_title('Edad según supervivencia y sexo')
axes[1].set_xlabel('Sobrevivió (0=No, 1=Sí)')
axes[1].set_ylabel('Edad (años)')

plt.suptitle('Análisis comparativo con boxplots', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Función de diagnóstico visual completo

In [ ]:
def diagnostico_visual(df, columnas, ncols=2):
    """Genera histograma + boxplot para cada variable numérica indicada."""
    nrows = -(-len(columnas) // ncols)  # ceil division
    fig, axes = plt.subplots(nrows * 2, ncols, figsize=(ncols * 6, nrows * 5))
    axes = axes.reshape(nrows * 2, ncols)

    for idx, col in enumerate(columnas):
        row_hist = (idx // ncols) * 2
        row_box = row_hist + 1
        col_idx = idx % ncols

        serie = df[col].dropna()

        # Histograma
        ax_h = axes[row_hist][col_idx]
        ax_h.hist(serie, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
        ax_h.axvline(serie.mean(), color='crimson', linestyle='--', linewidth=1.5,
                     label=f'Media: {serie.mean():.2f}')
        ax_h.axvline(serie.median(), color='darkorange', linestyle='-', linewidth=1.5,
                     label=f'Mediana: {serie.median():.2f}')
        ax_h.set_title(col, fontsize=11)
        ax_h.legend(fontsize=8)

        # Boxplot
        ax_b = axes[row_box][col_idx]
        ax_b.boxplot(serie, vert=False, patch_artist=True,
                     boxprops=dict(facecolor='steelblue', alpha=0.5),
                     medianprops=dict(color='darkorange', linewidth=2),
                     flierprops=dict(marker='o', markersize=3, alpha=0.4))
        ax_b.set_yticks([])

    # Ocultar ejes sobrantes
    for idx in range(len(columnas), nrows * ncols):
        axes[(idx // ncols) * 2][idx % ncols].set_visible(False)
        axes[(idx // ncols) * 2 + 1][idx % ncols].set_visible(False)

    plt.suptitle('Diagnóstico visual de dispersión', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

diagnostico_visual(titanic, ['age', 'fare', 'sibsp', 'parch'])

---
## 🔧 Ejercicios

### Ejercicio 1 — Dispersión en el dataset Penguins

Usando el dataset `penguins`, calcula la desviación típica, el IQR y el coeficiente de variación para las variables `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm` y `body_mass_g`. ¿Qué variable tiene más variabilidad relativa? ¿Cuál menos?

In [ ]:
# 1. Calcula std, IQR y CV para las 4 variables
# cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
# ...

# 2. Presenta los resultados en un DataFrame ordenado por CV descendente
# ...

# 3. ¿Qué variable descartarías por baja varianza? ¿Cuál escalarías con RobustScaler?

### Ejercicio 2 — Detección de outliers

Aplica la función `detectar_outliers_iqr()` a la variable `body_mass_g` del dataset `penguins` (sin nulos). ¿Cuántos outliers hay? ¿Son errores o valores legítimos? Visualízalo con un boxplot.

In [ ]:
# 1. Filtra nulos de body_mass_g y aplica detectar_outliers_iqr()
# masa = penguins['body_mass_g'].dropna()
# mask_out, lim_inf, lim_sup = detectar_outliers_iqr(masa)
# print(...)

# 2. Visualiza con boxplot
# ...

# 3. Investiga qué especie corresponde a los outliers
# print(penguins[mask_out][['species', 'body_mass_g']])

### Ejercicio 3 — Diagnóstico completo

Aplica la función `diagnostico_visual()` al dataset `diamonds` para las variables `price`, `carat`, `depth` y `table`. Identifica cuáles tienen distribuciones simétricas y cuáles asimétricas. Para las asimétricas, ¿qué transformación aplicarías para corregir la forma (logaritmo, raíz cuadrada...)?

In [ ]:
# 1. Aplica diagnostico_visual al dataset diamonds
# diagnostico_visual(diamonds, ['price', 'carat', 'depth', 'table'])

# 2. Anota qué variables tienen sesgo a la derecha o izquierda
# ...

# 3. Aplica np.log1p() a 'price' y 'carat' y compara los histogramas antes/después
# fig, axes = plt.subplots(2, 2, figsize=(12, 8))
# ...
# ¿Mejora la simetría? ¿Tiene sentido esta transformación?

---
## 📝 Resumen de la sesión

| Concepto | Clave |
|----------|-------|
| **Varianza / Std** | Miden dispersión alrededor de la media. Sensibles a outliers. |
| **Rango** | Distancia entre mín y máx. Muy sensible a extremos. |
| **IQR** | Rango del 50% central. Robusto ante outliers. Base del boxplot. |
| **Percentiles** | Dividen la distribución. P25, P50 y P75 son los cuartiles. |
| **Coeficiente de variación** | Dispersión relativa (%). Permite comparar variables con escalas distintas. |
| **Outlier IQR** | Valor fuera de [Q1 − 1.5·IQR, Q3 + 1.5·IQR]. |
| **Histograma** | Muestra la forma de la distribución y la concentración de valores. |
| **Boxplot** | Resume en 5 estadísticos. Ideal para comparar grupos y detectar outliers. |
| **Baja varianza** | Si CV ≈ 0, la variable apenas aporta al modelo. Candidata a descarte. |

### 🔜 Próxima sesión
**Bloque 2 — Sesión 1:** Distribuciones de probabilidad. Normal, uniforme y binomial. Sesgo y curtosis.